# 손 모양 모델 학습 — 1조 라즈봇 (지시서 `specs/08-hand-gesture.md`)

**HaGRID** 데이터셋에서 손 모양 4가지(`palm` 보 · `fist` 주먹 · `like` 엄지척 · `peace` 브이)와 명령 아닌 손(`no_gesture`)만 골라
**YOLO26n · 입력 320**으로 학습하고, 라즈봇 OpenCV에서 돌릴 **ONNX**까지 만든다.

### 쓰는 법
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**
2. 위에서부터 셀을 하나씩 실행 (▶). 각 셀은 끝나면 `완료`를 찍는다.
3. 마지막 셀이 `gestures_yolo26n_320.onnx` · `gestures_best.pt`를 내려받는다 → 노트북 `raspbot/models/`에 넣는다.

### 알아 둘 것
- HaGRID 전체는 1.5TB라 무료 Colab에 안 들어간다 → zip을 통째로 받지 않고 **필요한 사진만 골라 받는다** (서버가 부분 받기 지원).
- 받는 양: 사진 약 7,200장 × 약 1.3MB ≈ **9GB** + 라벨 약 0.4GB (추정). 걸리는 시간은 서버 속도에 따라 다름.
- 사진에는 얼굴이 있다 → **깃에 올리지 않는다.** 이 노트북 안(Colab)에서만 쓴다.
- 라이선스: HaGRID **CC BY-SA 4.0 변형** (https://github.com/hukenovs/hagrid/blob/master/license/en_us.pdf) · YOLO26 **AGPL-3.0**.
- Colab이 중간에 끊기면: 구글 드라이브에 저장하도록 두면(`SAVE_TO_DRIVE = True`) 「7. 학습」 셀 아래의 이어 하기 줄로 계속할 수 있다.

In [ ]:
# 1. 설정 — 지시서 08 「정해진 것」 · 「바꿔 볼 값」 (바꾸면 지시서부터)
GESTURES = ["palm", "fist", "like", "peace"]      # 손 모양 (조원 결정 2026-11-05)
CLASSES = GESTURES + ["no_gesture"]              # 라벨 번호 = 이 순서 (0 palm · 1 fist · 2 like · 3 peace · 4 no_gesture)
N_TRAIN, N_VAL = 1500, 300                       # 손 모양마다 학습 · 검증 장 수
IMG_MAX = 640                                    # 받은 사진을 긴 쪽 640으로 줄여 저장 (디스크 · 속도)
IMGSZ = 320                                      # 학습 입력 (라즈봇과 같게)
EPOCHS = 60
BATCH = 64
SEED = 0
THREADS = 8                                      # 동시에 받는 개수
SAVE_TO_DRIVE = True                             # 학습 결과를 구글 드라이브에 (끊겨도 남음)

IMG_URL = "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid/hagrid_dataset_new_554800/hagrid_dataset/{g}.zip"
ANN_URL = "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid_v2/annotations_with_landmarks/annotations.zip"
HEAD = {"User-Agent": "raspbot-pbl-1team-colab"}
ROOT = "/content/hagrid_yolo"
print("완료")

In [ ]:
# 2. GPU · 디스크 확인 + 도구 설치 (Colab 안에서만)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -n 1
%pip install -q ultralytics remotezip
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU", torch.cuda.is_available())
print("완료")

In [ ]:
# 3. 결과 저장 곳 (구글 드라이브를 쓰면 로그인 창이 뜬다)
import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/raspbot_gestures"
else:
    OUT = "/content/runs_gestures"
os.makedirs(OUT, exist_ok=True)
print("저장 곳:", OUT)
print("완료")

In [ ]:
# 4. 라벨(json)만 골라 받기 — annotations.zip(719MB)을 통째로 받지 않음
import json
from remotezip import RemoteZip
ann = {}
with RemoteZip(ANN_URL, headers=HEAD) as z:
    for split in ("train", "val"):
        for g in GESTURES:
            ann[(split, g)] = json.loads(z.read(f"annotations/{split}/{g}.json"))
            print(f"{split:5s} {g:6s} 사진 {len(ann[(split, g)]):,}장")
print("완료")

In [ ]:
# 5. 사진 골라 받기 → 긴 쪽 640으로 줄이기 → YOLO 라벨 쓰기
import io, os, random, time, concurrent.futures as cf
from collections import Counter
from PIL import Image
from remotezip import RemoteZip

CLASS_ID = {name: i for i, name in enumerate(CLASSES)}
for split in ("train", "val"):
    os.makedirs(f"{ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{ROOT}/labels/{split}", exist_ok=True)

random.seed(SEED)
jobs = []                                         # (손 모양, split, 사진 id)
for g in GESTURES:
    for split, n in (("train", N_TRAIN), ("val", N_VAL)):
        ids = sorted(ann[(split, g)])
        random.shuffle(ids)
        jobs += [(g, split, i) for i in ids[:n]]

def run_chunk(chunk):
    g = chunk[0][0]
    got, miss, boxes = 0, 0, Counter()
    with RemoteZip(IMG_URL.format(g=g), headers=HEAD) as z:
        names = set(z.namelist())
        for _, split, i in chunk:
            member = f"{g}/{i}.jpg"
            if member not in names:
                miss += 1
                continue
            img = Image.open(io.BytesIO(z.read(member))).convert("RGB")
            img.thumbnail((IMG_MAX, IMG_MAX))
            img.save(f"{ROOT}/images/{split}/{i}.jpg", quality=90)
            a = ann[(split, g)][i]
            lines = []
            for (x, y, w, h), lab in zip(a["bboxes"], a["labels"]):
                if lab in CLASS_ID:                   # 4가지 + no_gesture 만 (그 밖의 라벨은 버림)
                    lines.append(f"{CLASS_ID[lab]} {x + w / 2:.6f} {y + h / 2:.6f} {w:.6f} {h:.6f}")
                    boxes[lab] += 1
            with open(f"{ROOT}/labels/{split}/{i}.txt", "w") as f:
                f.write("\n".join(lines))
            got += 1
    return got, miss, boxes

chunks = []
for g in GESTURES:
    mine = [j for j in jobs if j[0] == g]
    chunks += [mine[k:k + 150] for k in range(0, len(mine), 150)]

t0, total, missed, all_boxes = time.time(), 0, 0, Counter()
with cf.ThreadPoolExecutor(THREADS) as ex:
    for n, (got, miss, boxes) in enumerate(ex.map(run_chunk, chunks), 1):
        total, missed = total + got, missed + miss
        all_boxes.update(boxes)
        print(f"\r{n}/{len(chunks)} 묶음 · 받음 {total:,}장 · zip에 없음 {missed} · {time.time() - t0:.0f}초", end="")
print()
print("상자 수:", dict(all_boxes))
if missed > len(jobs) * 0.05:
    print("!! zip에 없는 사진이 5%를 넘음 — 라벨과 사진 판이 다를 수 있음. 멈추고 확인")
!du -sh {ROOT}
print("완료")

In [ ]:
# 6. 학습 설정 파일 + 사진 한 장 확인 (상자가 손 위에 있어야 정상)
import yaml, glob, cv2
import matplotlib.pyplot as plt
with open(f"{ROOT}/data.yaml", "w") as f:
    yaml.safe_dump({"path": ROOT, "train": "images/train", "val": "images/val",
                    "names": {i: n for i, n in enumerate(CLASSES)}}, f, allow_unicode=True, sort_keys=False)
print(open(f"{ROOT}/data.yaml").read())

p = sorted(glob.glob(f"{ROOT}/images/train/*.jpg"))[0]
im = cv2.imread(p)
H, W = im.shape[:2]
for line in open(p.replace("/images/", "/labels/").replace(".jpg", ".txt")).read().split("\n"):
    if line:
        c, cx, cy, w, h = line.split()
        cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
        cv2.rectangle(im, (int(cx - w / 2), int(cy - h / 2)), (int(cx + w / 2), int(cy + h / 2)), (0, 255, 0), 3)
        cv2.putText(im, CLASSES[int(c)], (int(cx - w / 2), int(cy - h / 2) - 8), 0, 1, (0, 255, 0), 2)
plt.figure(figsize=(6, 6)); plt.imshow(im[:, :, ::-1]); plt.axis("off"); plt.show()
print("완료")

In [ ]:
# 7. 학습 — YOLO26n · 입력 320 (T4에서 수십 분 걸림 · 추정)
from ultralytics import YOLO
model = YOLO("yolo26n.pt")
model.train(data=f"{ROOT}/data.yaml", imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=15,
            workers=2, seed=SEED, project=OUT, name="gestures", exist_ok=True)
# Colab이 끊겼다면 1 · 2 · 3 셀만 다시 실행한 뒤 아래 줄의 # 를 지우고 실행:
# YOLO(f"{OUT}/gestures/weights/last.pt").train(resume=True)
print("완료")

In [ ]:
# 8. 검증 점수 (손 모양별 mAP50) — 지시서 08 성공 기준에 적는다
best = f"{OUT}/gestures/weights/best.pt"
m = YOLO(best)
r = m.val(data=f"{ROOT}/data.yaml", imgsz=IMGSZ, verbose=False)
print(f"전체 mAP50 {r.box.map50:.3f} · mAP50-95 {r.box.map:.3f}")
for i, name in m.names.items():
    print(f"  {i} {name:10s} mAP50 {r.box.ap50[i]:.3f}")
print("완료")

In [ ]:
# 9. 라즈봇용 ONNX 만들기 (06과 같은 옵션) + 지문 + 내려받기
import shutil, hashlib, onnx
onnx_path = YOLO(best).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, batch=1, dynamic=False)
dst_onnx, dst_pt = f"/content/gestures_yolo26n_{IMGSZ}.onnx", "/content/gestures_best.pt"
shutil.copy(onnx_path, dst_onnx)
shutil.copy(best, dst_pt)
g = onnx.load(dst_onnx)
meta = {p.key: p.value for p in g.metadata_props}
print("라벨:", meta.get("names"))
print("출력:", [[d.dim_value for d in o.type.tensor_type.shape.dim] for o in g.graph.output])
for p in (dst_onnx, dst_pt):
    print(p, os.path.getsize(p), "sha256", hashlib.sha256(open(p, "rb").read()).hexdigest())
shutil.copy(dst_onnx, OUT)
from google.colab import files
files.download(dst_onnx)
files.download(dst_pt)
print("완료 — 받은 두 파일을 노트북 raspbot/models/ 에 넣고 Claude에게 알려 주세요")